In [ ]:
import pandas as pd
import json
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import pytz
import requests
import re
import psycopg2
from functions import read_db_credentials, connect_to_db, load_json_data_from_db_as_json, save_df_to_db, data_from_data_sink

In [ ]:
# def read_db_credentials(path="data/config.txt"):
#     creds = {}
#     with open(path, "r") as f:
#         for line in f:
#             key, value = line.strip().split("=")
#             creds[key] = value
#     return creds


# def connect_to_db(creds):
#     return psycopg2.connect(
#         host=creds["host"],
#         port=creds["port"],
#         dbname=creds["database"],
#         user=creds["user"],
#         password=creds["password"]
#     )
# def data_from_data_sink(query):
#     creds = read_db_credentials()
#     conn = connect_to_db(creds)
#     cur = conn.cursor()
    
#     df = pd.read_sql_query(query, conn)
    
#     conn.close()
    
#     return df

# def save_df_to_db(df, table_name):
#     creds = read_db_credentials()
#     conn = connect_to_db(creds)
#     cursor = conn.cursor()
    
#     df = df.where(pd.notnull(df), None)  # NaN → None
    
#     columns = ', '.join(df.columns)
#     placeholders = ', '.join(['%s'] * len(df.columns))
    
#     insert_query = f"""
#         INSERT INTO {table_name} ({columns})
#         VALUES ({placeholders})
#     """
    
#     for _, row in df.iterrows():
#         for i, val in enumerate(row.tolist()):
#             print(f"{df.columns[i]} -> {val} ({type(val)})")

#         cursor.execute(insert_query, row.tolist())


#     conn.commit()
#     cursor.close()
#     conn.close()
#     return ("saved!")
    
# def load_json_data_from_db_as_json(user, source ):
#     creds = read_db_credentials()
#     conn = connect_to_db(creds)

#     query = f"SELECT raw_json FROM fact_raw_data WHERE data_source = '{source}' AND user_number = {user} ;"
    
#     cursor = conn.cursor()
#     cursor.execute(query)
#     result = cursor.fetchone()
#     conn.close()
#     #return (result)
#     # # Parsen der JSON-Inhalte aus der 'data'-Spalte
#     parsed_data = result[0] if result else {}


#     # # Rückgabe als JSON-String (optional indent für Lesbarkeit)
#     return parsed_data


In [ ]:
with open("data/cur_user_selected.txt", "r", encoding="utf-8") as f:
    user = int(f.read().strip())

git_data = load_json_data_from_db_as_json(user, "github")
print(git_data)

{'username': 'GokanGorer', 'name': None, 'bio': None, 'location': None, 'company': None, 'website': None, 'emails': ['s_goerer20@stud.hwr-berlin.de'], 'public_repos': 5, 'repo_names': ['Docker', 'gapminder', 'workwithapi', 'fastapi', 'kaftkaroland'], 'avatar_url': 'https://avatars.githubusercontent.com/u/207158506?v=4', 'html_url': 'https://github.com/GokanGorer', 'created_at': '2025-04-11T08:22:20Z', 'plan': 'free'}


In [14]:
# Top-Level Keys anzeigen
data = git_data
import json

def print_json_structure(data, indent=0):
    spacer = "  " * indent
    if isinstance(data, dict):
        for key, value in data.items():
            print(f"{spacer}\"{key}\": ", end="")
            if isinstance(value, (dict, list)):
                print()
                print_json_structure(value, indent + 1)
            else:
                print(type(value).__name__)
    elif isinstance(data, list):
        print(f"{spacer}[")
        if data:
            print_json_structure(data[0], indent + 1)
        else:
            print(f"{'  ' * (indent + 1)}<empty>")
        print(f"{spacer}]")
    else:
        print(f"{spacer}{type(data).__name__}")

# JSON-Datei laden

# Struktur ausgeben
print_json_structure(data)


"username": str
"name": NoneType
"bio": NoneType
"location": NoneType
"company": NoneType
"website": NoneType
"emails": 
  [
    str
  ]
"public_repos": int
"repo_names": 
  [
    str
  ]
"avatar_url": str
"html_url": str
"created_at": str
"plan": str


In [15]:
def categorize_github_coding_behavior(data: dict) -> str:
    """
    Categorizes a GitHub user's coding activity based on actual repository usage,
    not just the public_repos count.

    Categories:
    - "Active Coder": Has 10 or more repositories listed in 'repo_names'
    - "Time to Time Coder": Has 1–9 repositories listed
    - "No Coder": Has no repositories listed (empty, null, or missing 'repo_names')

    :param data: Dictionary containing GitHub user data with a 'repo_names' field
    :return: A string classification of coding behavior
    """
    repo_names = data.get("repo_names", [])

    # Ensure it's a list
    if not isinstance(repo_names, list) or len(repo_names) == 0:
        return "No Coder"

    repo_count = len(repo_names)

    if repo_count >= 10:
        return "Active Coder"
    elif repo_count >= 1:
        return "Time to Time Coder"
    else:
        return "No Coder"


# Coding TYPE

In [16]:
coding_type = categorize_github_coding_behavior(data)
print(coding_type)

Time to Time Coder


# REPO COUNTS

In [17]:
repo_names = data.get("repo_names", [])
repo_count= len(repo_names)

# SKILLS

In [18]:
def extract_top_languages(data: dict):
    """
    Extracts the top 3 most used programming languages by percentage from a language usage dictionary.

    Returns:
    - code_lang_1: "Language (X%)"
    - code_lang_2: "Language (X%)"
    - code_lang_3: "Language (X%)"

    If fewer than 3 languages exist, fills remaining slots with "NA".
    """
    lang_stats = data.get("language_stats", {})
    if not lang_stats:
        return "NA", "NA", "NA"

    # Sort by percentage descending
    sorted_langs = sorted(lang_stats.items(), key=lambda x: x[1], reverse=True)

    # Build formatted strings
    formatted = [f"{lang} ({round(percent)}%)" for lang, percent in sorted_langs[:3]]

    # Pad with "NA" if fewer than 3
    while len(formatted) < 3:
        formatted.append("NA")

    return tuple(formatted)


In [21]:

res = pd.DataFrame([{
    "user_number": user,
    "coding_type": coding_type,
    "repo_count": repo_count
}])
print(res)

save_df_to_db(res, "dim_github")

   user_number         coding_type  repo_count
0            8  Time to Time Coder           5
user_number -> 8 (<class 'int'>)
coding_type -> Time to Time Coder (<class 'str'>)
repo_count -> 5 (<class 'int'>)


'saved!'